In [ ]:
# Install the Python packages required by this notebook.
%pip install -q duckdb pyarrow scikit-learn xgboost scipy joblib

In [ ]:
# Mount Google Drive so this notebook can access the private MIMIC-IV data and derived files.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Define the MIMIC-IV folders and the shared derived-data folder used by all notebooks.
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/Early Acute Kidney Injury Prediction + Production Monitoring/data")
HOSP_DIR = DATA_ROOT / "hosp"
ICU_DIR = DATA_ROOT / "icu"
DERIVED_DIR = DATA_ROOT / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the libraries used to measure temporal feature drift and model-performance degradation.
import json
import joblib
import numpy as np
import pandas as pd

from scipy.stats import ks_2samp
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [ ]:
# Load the historical reference cohort, later current cohort, trained model, feature list, and decision threshold.
MODEL_DIR = DERIVED_DIR / "models"
EVALUATION_DIR = DERIVED_DIR / "evaluation"

reference_df = pd.read_parquet(DERIVED_DIR / "train.parquet")
current_df = pd.read_parquet(DERIVED_DIR / "test.parquet")

xgboost_model = joblib.load(MODEL_DIR / "xgboost.joblib")

with open(MODEL_DIR / "feature_columns.json") as file:
    feature_columns = json.load(file)

with open(EVALUATION_DIR / "thresholds.json") as file:
    xgboost_threshold = json.load(file)["xgboost"]

In [ ]:
# Define PSI and KS-based helpers for monitoring numeric feature-distribution and missingness drift.
def population_stability_index(reference, current, bins=10):
    reference = pd.Series(reference).dropna().astype(float)
    current = pd.Series(current).dropna().astype(float)

    if len(reference) == 0 or len(current) == 0:
        return np.nan

    edges = np.unique(
        np.quantile(reference, np.linspace(0, 1, bins + 1))
    )

    if len(edges) < 3:
        return 0.0

    edges[0] = -np.inf
    edges[-1] = np.inf

    reference_counts, _ = np.histogram(reference, bins=edges)
    current_counts, _ = np.histogram(current, bins=edges)

    reference_pct = np.clip(
        reference_counts / max(reference_counts.sum(), 1),
        1e-6,
        None,
    )
    current_pct = np.clip(
        current_counts / max(current_counts.sum(), 1),
        1e-6,
        None,
    )

    return float(
        np.sum(
            (current_pct - reference_pct)
            * np.log(current_pct / reference_pct)
        )
    )

def numeric_drift_table(reference_df, current_df, columns):
    rows = []

    for column in columns:
        reference = reference_df[column]
        current = current_df[column]

        reference_non_null = reference.dropna()
        current_non_null = current.dropna()

        if len(reference_non_null) and len(current_non_null):
            ks_result = ks_2samp(
                reference_non_null,
                current_non_null,
            )
            ks_statistic = float(ks_result.statistic)
            ks_pvalue = float(ks_result.pvalue)
        else:
            ks_statistic = np.nan
            ks_pvalue = np.nan

        rows.append(
            {
                "feature": column,
                "reference_missing_rate": float(reference.isna().mean()),
                "current_missing_rate": float(current.isna().mean()),
                "reference_mean": float(reference_non_null.mean()) if len(reference_non_null) else np.nan,
                "current_mean": float(current_non_null.mean()) if len(current_non_null) else np.nan,
                "psi": population_stability_index(reference, current),
                "ks_statistic": ks_statistic,
                "ks_pvalue": ks_pvalue,
            }
        )

    result = pd.DataFrame(rows)

    result["drift_flag"] = (
        (result["psi"] >= 0.20)
        | (result["ks_pvalue"] < 0.01)
        | (
            (
                result["current_missing_rate"]
                - result["reference_missing_rate"]
            ).abs()
            >= 0.10
        )
    )

    return result.sort_values(
        ["drift_flag", "psi"],
        ascending=[False, False],
    )

In [ ]:
# Calculate feature-level drift between the earlier reference cohort and the later test cohort.
numeric_columns = [
    column
    for column in feature_columns
    if pd.api.types.is_numeric_dtype(reference_df[column])
]

feature_drift = numeric_drift_table(
    reference_df,
    current_df,
    numeric_columns,
)

feature_drift.head(20)

In [ ]:
# Define the binary performance metrics used to compare reference and current model behavior.
def performance_metrics(y_true, y_probability, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_probability = np.asarray(y_probability)
    y_prediction = (y_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_prediction,
        labels=[0, 1],
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "n": int(len(y_true)),
        "prevalence": float(y_true.mean()),
        "auroc": float(roc_auc_score(y_true, y_probability)),
        "auprc": float(average_precision_score(y_true, y_probability)),
        "sensitivity": float(recall_score(y_true, y_prediction, zero_division=0)),
        "specificity": float(specificity),
        "ppv": float(precision_score(y_true, y_prediction, zero_division=0)),
        "brier": float(brier_score_loss(y_true, y_probability)),
    }

In [ ]:
# Compare model predictions and clinical performance between the historical reference and later current cohorts.
reference_probability = xgboost_model.predict_proba(
    reference_df[feature_columns]
)[:, 1]

current_probability = xgboost_model.predict_proba(
    current_df[feature_columns]
)[:, 1]

reference_metrics = performance_metrics(
    reference_df["target"],
    reference_probability,
    xgboost_threshold,
)

current_metrics = performance_metrics(
    current_df["target"],
    current_probability,
    xgboost_threshold,
)

pd.DataFrame(
    [reference_metrics, current_metrics],
    index=["Reference", "Current"],
)

In [ ]:
# Create a simple monitoring status from sensitivity degradation and the number of drifted numeric features.
sensitivity_drop = (
    reference_metrics["sensitivity"]
    - current_metrics["sensitivity"]
)

drifted_feature_count = int(
    feature_drift["drift_flag"].sum()
)

if current_metrics["sensitivity"] < 0.80 or sensitivity_drop >= 0.10:
    monitoring_status = "CRITICAL"
elif drifted_feature_count >= 5:
    monitoring_status = "WARNING"
else:
    monitoring_status = "HEALTHY"

monitoring_summary = {
    "status": monitoring_status,
    "reference_metrics": reference_metrics,
    "current_metrics": current_metrics,
    "reference_mean_prediction": float(reference_probability.mean()),
    "current_mean_prediction": float(current_probability.mean()),
    "sensitivity_drop": float(sensitivity_drop),
    "drifted_numeric_features": drifted_feature_count,
    "numeric_features_monitored": int(len(feature_drift)),
}

monitoring_summary

In [ ]:
# Save the drift table and aggregate monitoring summary without exporting patient-level MIMIC data.
MONITORING_DIR = DERIVED_DIR / "monitoring"
MONITORING_DIR.mkdir(parents=True, exist_ok=True)

feature_drift.to_csv(
    MONITORING_DIR / "feature_drift.csv",
    index=False,
)

with open(MONITORING_DIR / "monitoring_summary.json", "w") as file:
    json.dump(
        monitoring_summary,
        file,
        indent=2,
    )

print("Saved monitoring outputs.")